# 01 - Data Profiling: FDA Medical Device Master Data
## Project: Healthcare Device Master Data Governance
### Objective
Perform initial data profiling on the FDA 510(k) Premarket Notification dataset 
to understand structure, quality issues, and MDM challenges before governance 
processes are applied.

In [1]:
# Import libraries
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [4]:
# Load the FDA 510(k) dataset
df = pd.read_csv('/Users/harshithasunkara/Documents/Projects/Healthcare MDM/data/raw/pmn96cur.txt', 
                 sep='|', 
                 encoding='latin-1', 
                 low_memory=False)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")

Dataset shape: (99216, 22)

Column names:
['KNUMBER', 'APPLICANT', 'CONTACT', 'STREET1', 'STREET2', 'CITY', 'STATE', 'COUNTRY_CODE', 'ZIP', 'POSTAL_CODE', 'DATERECEIVED', 'DECISIONDATE', 'DECISION', 'REVIEWADVISECOMM', 'PRODUCTCODE', 'STATEORSUMM', 'CLASSADVISECOMM', 'SSPINDICATOR', 'TYPE', 'THIRDPARTY', 'EXPEDITEDREVIEW', 'DEVICENAME']


In [5]:
# Basic dataset overview
print("=== DATASET OVERVIEW ===")
print(f"Total Records: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")

print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== FIRST 5 ROWS ===")
df.head()

=== DATASET OVERVIEW ===
Total Records: 99,216
Total Columns: 22

=== DATA TYPES ===
KNUMBER              object
APPLICANT            object
CONTACT              object
STREET1              object
STREET2              object
CITY                 object
STATE                object
COUNTRY_CODE         object
ZIP                  object
POSTAL_CODE          object
DATERECEIVED         object
DECISIONDATE         object
DECISION             object
REVIEWADVISECOMM     object
PRODUCTCODE          object
STATEORSUMM          object
CLASSADVISECOMM      object
SSPINDICATOR        float64
TYPE                 object
THIRDPARTY           object
EXPEDITEDREVIEW      object
DEVICENAME           object
dtype: object

=== FIRST 5 ROWS ===


,KNUMBER,APPLICANT,CONTACT,STREET1,STREET2,CITY,STATE,COUNTRY_CODE,ZIP,POSTAL_CODE,DATERECEIVED,DECISIONDATE,DECISION,REVIEWADVISECOMM,PRODUCTCODE,STATEORSUMM,CLASSADVISECOMM,SSPINDICATOR,TYPE,THIRDPARTY,EXPEDITEDREVIEW,DEVICENAME
0,DEN000001,Ohmeda Medical,DANIEL KOSEDNAR,P.O. Box 7550,NaN,Madison,WI,US,53707,53707,01/07/2000,01/11/2000,DENG,AN,MRN,NaN,AN,NaN,Post-NSE,N,NaN,OHMEDA INOVENT DELIVERY SYSTEM
1,K000001,"Boston Scientific Scimed, Inc.",RON BENNETT,5905 Nathan Ln.,NaN,Plymouth,MN,US,55442,55442,01/03/2000,06/05/2000,SESE,SU,JCT,Summary,SU,NaN,Traditional,N,NaN,WALLGRAFT TRACHEOBRONCHIAL ENDOPROSTHESIS AND ...
2,DEN000002,"Urosurge, Inc.",STEVEN J PREISS,2660 Crosspark Rd.,NaN,Coralville,IA,US,52241,52241,01/27/2000,02/09/2000,DENG,GU,NAM,NaN,GU,NaN,Post-NSE,N,NaN,UROSURGE PERCUTANEOUS SANS (STOLLER AFFERENT N...
3,K000002,"Usa Instruments, Inc.",RONY THOMAS,1515 Danner Dr.,NaN,Aurora,OH,US,44202,44202,01/03/2000,02/23/2000,SESE,RA,MOS,Summary,RA,NaN,Traditional,N,NaN,MAGNA 5000 PHASED ARRAY CTL SPINE COIL
4,K000003,Tornier,DAVID W SCHLERF,200 Gregory Ln. Suite C-100,NaN,Pleasant Hill,CA,US,94523,94523,01/03/2000,10/02/2000,SESE,OR,JDC,Summary,OR,NaN,Traditional,N,NaN,TORNIER TOTAL ELBOW PROSTHESIS


In [6]:
# Missing values analysis
print("=== MISSING VALUES ANALYSIS ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print(missing_df)

=== MISSING VALUES ANALYSIS ===
                  Missing Count  Missing %
SSPINDICATOR              99216     100.00
EXPEDITEDREVIEW           99188      99.97
STREET2                   81519      82.16
STATE                     25246      25.45
CLASSADVISECOMM            2115       2.13
ZIP                        1722       1.74
POSTAL_CODE                1722       1.74
STATEORSUMM                 959       0.97
CONTACT                     147       0.15
STREET1                      79       0.08
PRODUCTCODE                   8       0.01
KNUMBER                       0       0.00
THIRDPARTY                    0       0.00
TYPE                          0       0.00
DECISIONDATE                  0       0.00
REVIEWADVISECOMM              0       0.00
DECISION                      0       0.00
APPLICANT                     0       0.00
DATERECEIVED                  0       0.00
COUNTRY_CODE                  0       0.00
CITY                          2       0.00
DEVICENAME            

In [7]:
# Duplicate detection - core MDM concept
print("=== DUPLICATE ANALYSIS ===")

# Full row duplicates
full_dupes = df.duplicated().sum()
print(f"Fully duplicate rows: {full_dupes}")

# Duplicate KNUMBERs (should be unique - it's our master key)
knumber_dupes = df['KNUMBER'].duplicated().sum()
print(f"Duplicate KNUMBERs: {knumber_dupes}")

# Duplicate device names (same device, different entries?)
device_dupes = df['DEVICENAME'].duplicated().sum()
print(f"Duplicate DEVICENAME entries: {device_dupes}")

# Applicant name variations (classic MDM problem)
print(f"\nUnique Applicants: {df['APPLICANT'].nunique():,}")
print(f"\nSample Applicant names (first 20):")
print(df['APPLICANT'].value_counts().head(20))

=== DUPLICATE ANALYSIS ===
Fully duplicate rows: 0
Duplicate KNUMBERs: 0
Duplicate DEVICENAME entries: 8150

Unique Applicants: 23,272

Sample Applicant names (first 20):
APPLICANT
Siemens Medical Solutions USA, Inc.    615
Smith & Nephew, Inc.                   486
Synthes (Usa)                          360
Dade Behring, Inc.                     341
C.R. Bard, Inc.                        333
Biomet, Inc.                           332
Boston Scientific Corp                 316
Arthrex, Inc.                          315
Howmedica Osteonics Corp.              288
Wrightmedicaltechnologyinc             274
Beckman Coulter, Inc.                  268
Abbott Laboratories                    267
Roche Diagnostics Corp.                264
Zimmer, Inc.                           254
Boston Scientific Corporation          228
Medtronic, Inc.                        207
DePuy Orthopaedics, Inc.               206
STERIS Corporation                     204
Becton, Dickinson & CO                 190
Me

In [8]:
# Show applicant name variation examples - classic MDM problem
print("=== APPLICANT NAME VARIATIONS (MDM Deduplication Problem) ===")
boston = df[df['APPLICANT'].str.contains('Boston Scientific', case=False, na=False)]
print(f"'Boston Scientific' variations:\n{boston['APPLICANT'].value_counts()}")

print("\n")
beckman = df[df['APPLICANT'].str.contains('Beckman', case=False, na=False)]
print(f"'Beckman' variations:\n{beckman['APPLICANT'].value_counts()}")

=== APPLICANT NAME VARIATIONS (MDM Deduplication Problem) ===
'Boston Scientific' variations:
APPLICANT
Boston Scientific Corp                                          316
Boston Scientific Corporation                                   228
Boston Scientific                                                58
Boston Scientific Scimed, Inc.                                   28
Boston Scientific, Target                                        19
Meadox Medicals, Div. Boston Scientific Corp.                    10
Boston Scientific Neuromodulation                                 9
Boston Scientific/Medi-Tech                                       5
Boston Scientific /International Technologies                     4
Boston Scientific - Precision Vascular                            3
Boston Scientific Ivt                                             2
Boston Scientific Corporation Northwest Technology                2
Boston Scientific Epi                                             2
Nxthera (A B

In [9]:
# MDM Profiling Summary
print("=== MDM DATA PROFILING SUMMARY ===")
print(f"""
DATASET: FDA 510(k) Premarket Notification
TOTAL RECORDS: {df.shape[0]:,}
TOTAL FIELDS: {df.shape[1]}

KEY MDM FINDINGS:
-----------------------------------------
1. COMPLETENESS ISSUES:
   - SSPINDICATOR: 100% missing (drop candidate)
   - EXPEDITEDREVIEW: 99.97% missing (drop candidate)
   - STATE: 25.45% missing (critical gap)
   - DEVICENAME: 2 missing (master data field)

2. UNIQUENESS ISSUES:
   - Duplicate KNUMBERs: 0 (master key is clean)
   - Duplicate DEVICENAME entries: 8,150

3. CONSISTENCY ISSUES:
   - Unique APPLICANT variations: 23,272
   - 'Boston Scientific' alone has 21 name variations
   - 'Beckman Coulter' has 4+ name variations
   - Inconsistent formatting (spaces, abbreviations, legal suffixes)

4. MDM ACTION ITEMS:
   - Drop SSPINDICATOR and EXPEDITEDREVIEW columns
   - Standardize APPLICANT names → create golden records
   - Resolve DEVICENAME duplicates
   - Impute or flag missing STATE values
""")

=== MDM DATA PROFILING SUMMARY ===

DATASET: FDA 510(k) Premarket Notification
TOTAL RECORDS: 99,216
TOTAL FIELDS: 22

KEY MDM FINDINGS:
-----------------------------------------
1. COMPLETENESS ISSUES:
   - SSPINDICATOR: 100% missing (drop candidate)
   - EXPEDITEDREVIEW: 99.97% missing (drop candidate)
   - STATE: 25.45% missing (critical gap)
   - DEVICENAME: 2 missing (master data field)

2. UNIQUENESS ISSUES:
   - Duplicate KNUMBERs: 0 (master key is clean)
   - Duplicate DEVICENAME entries: 8,150

3. CONSISTENCY ISSUES:
   - Unique APPLICANT variations: 23,272
   - 'Boston Scientific' alone has 21 name variations
   - 'Beckman Coulter' has 4+ name variations
   - Inconsistent formatting (spaces, abbreviations, legal suffixes)

4. MDM ACTION ITEMS:
   - Drop SSPINDICATOR and EXPEDITEDREVIEW columns
   - Standardize APPLICANT names → create golden records
   - Resolve DEVICENAME duplicates
   - Impute or flag missing STATE values

